In [18]:
import sys
sys.path.append("..")

In [19]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [20]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [21]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    theta_adv = deepcopy(theta_0)
    # theta_adv[-1] -= alpha
    # for i in range(X_0.shape[1]):
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_adv)
        theta_r_max = deepcopy(theta_adv)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        if np.mean(J_min) >= np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha
    
    return theta_adv

In [22]:
def get_theta_adv_linf_wc(x_r, weights_0, bias_0, alpha):
    weights_adv, bias_adv = deepcopy(weights_0), deepcopy(bias_0)
    weights_adv = weights_0 - (alpha * np.sign(x_r))
    for i in range(len(x_r)):
        if np.sign(x_r[i]) == 0:
            weights_adv[i] = weights_0[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias_0 - alpha
        
    return weights_adv, bias_adv

In [ ]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_type='avg'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]
    
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    if theta_adv_type == 'avg':
        theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    
        weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        clf_adv = deepcopy(clf)
        clf_adv.model.coef_ = weights_adv.reshape(1,-1)
        clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        if theta_adv_type == 'abs':
            weights_adv, bias_adv = get_theta_adv_linf_wc(x_r, weights_0, bias_0, alpha)
            theta_adv = np.hstack((weights_adv, bias_adv))
            clf_adv = deepcopy(clf)
            clf_adv.model.coef_ = weights_adv.reshape(1,-1)
            clf_adv.model.intercept_ = bias_adv
        
        J = RecourseCost(x_0, lamb)
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [24]:
params = {}
# 'synthetic', 'german', 'sba'
params['data'] = 'synthetic'
# 'lr', 'nn'
params['base_model'] = 'lr'
params['seeds'] = range(5)
# TODO: add your method here, the method name should match the name in filepath
params['algorithms'] = ['Alg1', 'ROAR']
params['theta_adv_type'] = 'abs'

alphas = np.arange(0.02, 0.52, 0.02).round(2)
lambdas = [0.05, 0.1, 0.2, 0.3]

results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'J': []
}

for algorithm in params['algorithms']:
    for lamb in lambdas:
        for alpha in alphas:
            for seed in params['seeds']:
                data = pd.read_pickle(f"../results/recourse-2025_08/{params['base_model']}_{params['data']}_{algorithm}_{lamb}_{alpha}_{seed}.pkl")
                theta_0 = data["theta_0"].iloc[0]
                weights_0, bias_0, = theta_0[:-1], theta_0[[-1]]
                X_0 = np.stack(data["x_0"])
                X_r = np.stack(data["x_r"])
                res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, params['theta_adv_type'])
                results['algorithm'].append(algorithm)
                results['seed'].append(seed)
                results['alpha'].append(alpha)
                results['lambda'].append(lamb)
                results['Cost'].append(res['cost'])
                results['Current Validity'].append(res['m1_probability'])
                results['Worst Case Validity'].append(res['wc_probability'])
                results['J'].append(res['J'])

df_results = pd.DataFrame(results)

[Alg1] [ seed=0 ] [ α=0.02 ] [ λ=0.05 ]: 100%|██████████| 96/96 [00:00<00:00, 2093.05it/s]


UnboundLocalError: cannot access local variable 'theta_adv' where it is not associated with a value

In [ ]:
df_results

,algorithm,seed,alpha,lambda,Cost,Current Validity,Worst Case Validity,J
0,Alg1,0,0.02,0.05,5.878046,0.975758,0.974373,0.319863
1,Alg1,1,0.02,0.05,5.809360,0.977444,0.974305,0.316499
2,Alg1,2,0.02,0.05,5.887283,0.975516,0.974118,0.320587
3,Alg1,3,0.02,0.05,5.930806,0.977772,0.974538,0.322332
4,Alg1,4,0.02,0.05,5.952183,0.977815,0.974558,0.323380
...,...,...,...,...,...,...,...,...
995,ROAR,0,0.50,0.30,4.788333,0.820628,0.658718,1.856024
996,ROAR,1,0.50,0.30,4.723372,0.818563,0.659395,1.835963
997,ROAR,2,0.50,0.30,4.782766,0.817244,0.656037,1.858719
998,ROAR,3,0.50,0.30,4.834855,0.812982,0.649532,1.883363


In [ ]:
print(f'{params["data"]}  |  {params["base_model"].upper()}')
df_results_avg = df_results.groupby(['algorithm', 'lambda'], as_index=False).mean(True)
df_results_im = df_results_avg.copy()
df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']] = df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str) + '±' + df_results.groupby(['algorithm', 'lambda'], as_index=False).std(numeric_only=True)[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str)

df_results_im

synthetic  |  LR


,algorithm,lambda,seed,alpha,Cost,Current Validity,Worst Case Validity,J
0,Alg1,0.05,2.0,0.26,6.24±0.23,0.99±0.01,0.97±0.0,0.34±0.01
1,Alg1,0.10,2.0,0.26,5.81±0.19,0.97±0.01,0.94±0.01,0.64±0.02
2,Alg1,0.20,2.0,0.26,5.36±0.15,0.93±0.02,0.88±0.01,1.2±0.04
3,Alg1,0.30,2.0,0.26,5.08±0.13,0.89±0.02,0.82±0.02,1.72±0.06
4,ROAR,0.05,2.0,0.26,5.21±0.14,0.91±0.02,0.85±0.01,0.42±0.02
5,ROAR,0.10,2.0,0.26,5.08±0.13,0.88±0.02,0.82±0.02,0.71±0.03
6,ROAR,0.20,2.0,0.26,4.87±0.11,0.84±0.02,0.76±0.02,1.25±0.05
7,ROAR,0.30,2.0,0.26,4.71±0.1,0.79±0.02,0.71±0.03,1.76±0.06


In [ ]:
df_results_avg = df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean()

In [ ]:
data_map = {'synthetic': 'Synthetic', 'sba': 'Small Business Administration', 'german': 'German', 'income': 'ACS Income'}
model_map = {'lr': 'Logistic Regression', 'nn': 'Neural Network'}

# colors = ['#1f77b4', '#17becf', '#9467bd', '#e377c2', '#2ca02c'] # Synthesis
colors = ['#C7E8F0', "#7FCBDC", "#37AEC8", '#236F80', '#E2C2F4', "#BD74E7", "#9726D9", "#61188B"] # Synthesis
# colors = ['#17becf', '#e377c2', '#2ca02c'] # German
# colors = ['#17becf', '#9467bd', '#e377c2', '#2ca02c'] # SBA

nc = len(colors)
font_family = 'Times New Roman'
font_color = 'black'
width, height = 720, 540

symbols = ['x' for _ in range(len(lambdas))] + ['circle']
size = [7 for _ in range(len(lambdas))] + [5]

fig = go.Figure()
c = 0
for i, alg in enumerate(params["algorithms"]):
    for lamb in lambdas:
        df_alg = df_results_avg.copy()
        df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['lambda']==lamb)]
        # df_alg = df_results[(df_results['algorithm'] == alg) & (df_results['lambda']==lamb) & (df_results['alpha']<=0.2)].sort_values(['Cost'], ascending=True).copy().reset_index(drop=True)
        # x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
        # df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask]})
        x, y = df_alg['Cost'], df_alg['Worst Case Validity']
        df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

        fig.add_trace(go.Scatter(
            x = df_alg['Cost'],
            y = df_alg['Worst Case Validity'],
            marker = dict(color=colors[c], size=3),
            mode = 'lines+markers' if alg != 'wachter' else 'markers',
            name = f"{alg} (λ={lamb})",
            showlegend=True,
            customdata=df_alg['alpha'],
            hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata}'
        ))
        c+=1

fig.update_xaxes(
    title=dict(
        text='Cost',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        )
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey', 
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_yaxes(
    title=dict(
        text='Worst Case Validity',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        ), 
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey',
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_layout(
    width=width,
    height=height,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=50,b=25,l=25,r=25),
    title =dict(
        # text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | Average Adversary", 
        text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC Adversary", 
        x= 0.5, 
        font=dict(family=font_family, size=20)
        ),
    legend=dict(
        x=0.975, 
        y=0.025, 
        orientation='v',
        xanchor='right',
        font=dict(
            family=font_family,
            color=font_color,
            size=15
            ), 
        bgcolor='rgba(255, 255, 255, 0.7)',
        bordercolor='lightgrey',
        borderwidth=1,
        entrywidth=100.5,
        ),
    xaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20,
        ),
    ),
    yaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20
        ),
        range=[-0.1,1.1],
    )
)

print(f'{params["data"]}  |  {params["base_model"].upper()}')
fig.show()

synthetic  |  LR


In [ ]:
# fig.write_image(f"../figs/cost_validity_{params['data']}_avgadv.png", engine='kaleido')